# CPU-1: Byte-Level LM - Training en 2xT4

Entrena CPU-1 (~170M parametros ternarios) con distillation desde Qwen-14B.

**Requisito:** Settings -> Accelerator -> GPU T4 x2

Si Kaggle asigna P100 (API push), el notebook escribe los archivos
pero saltea el entrenamiento. Correr manualmente con T4 x2 desde la UI.

In [ ]:
import os, sys, subprocess, pathlib, re

# Detectar GPUs ANTES de importar torch
r = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap",
                    "--format=csv,noheader"], capture_output=True, text=True)
gpu_lines = [l.strip() for l in r.stdout.strip().split("\n") if l.strip()]
n_gpus = len(gpu_lines)

print("Python:", sys.version.split()[0])
print("GPUs detectadas:", n_gpus)

GPU_OK = False
NEED_DOWNGRADE = False

for i, line in enumerate(gpu_lines):
    parts = line.split(", ")
    name = parts[0]
    cc = float(parts[1]) if len(parts) > 1 else 0
    ok = "OK" if cc >= 7.0 else "INCOMPATIBLE"
    print(f"  GPU {i}: {name} (cc {cc}) [{ok}]")
    if cc < 7.0:
        NEED_DOWNGRADE = True

if n_gpus == 0:
    print("ERROR: No se detectaron GPUs -> Settings -> GPU T4 x2")
elif NEED_DOWNGRADE:
    print()
    print("=" * 60)
    print("GPU(s) con compute capability < 7.0 (P100 = sm_60)")
    print("PyTorch moderno de Kaggle no soporta esta GPU.")
    print()
    print("ACCION REQUERIDA:")
    print("  1. Abrir este notebook en la UI de Kaggle")
    print("  2. Settings -> Accelerator -> GPU T4 x2")
    print("  3. Click Save & Run All (Commit)")
    print()
    print("Los archivos se escribiran pero el dry-run se saltea.")
    print("=" * 60)
else:
    GPU_OK = True
    if n_gpus >= 2:
        print("OK: 2 GPUs compatibles, listas para DDP")
    else:
        print("OK: GPU compatible (single-GPU mode)")


In [ ]:
import torch

if GPU_OK:
    print("PyTorch:", torch.__version__)
    print("CUDA   :", torch.version.cuda)
    # Test rapido
    try:
        t = torch.zeros(1, device="cuda:0")
        del t
        print("OK: CUDA funcional")
    except Exception as e:
        print("ERROR CUDA:", e)
        GPU_OK = False
else:
    print("PyTorch:", torch.__version__, "(GPU no compatible, solo escritura de archivos)")


## Escribir archivos del modelo y entrenamiento

In [ ]:
%%writefile /kaggle/working/model_v2.py
"""
CPU-1: ~170M parameter byte-level language model with ternary weights, MatMul-free
token mixing, and PFNet (Prefetch-Aware Nonlinear Residual).

Arquitectura:
- BytePatchEmbeddingV2: BitEmbedding ternario (4 bytes -> 1024 dims)
- MatMulFreeTokenMixer: MLGRU sin multiplicacion de matrices (Hadamard only)
- PFNet: 2-layer ternary MLP residual branch (224 KB total, L2-resident)
- BitHead: prediction heads ternarios
- KV cache: eliminado — estado MLGRU de 28KB fijo

Fuentes:
- Slender-Mamba (COLING 2025): cuantizacion de embeddings y heads
- MatMul-free LM (arXiv 2406.02528): MLGRU sin multiplicacion de matrices

El objetivo es maximizar tokens/segundo en CPU de consumo (4 cores, 16GB RAM, 32MB L3).
"""

import ctypes
import math
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor


# =============================================================================
# Configuracion global v2
# =============================================================================

D_MODEL = 1024
N_LAYERS = 14
D_FF = 2730
FP_RANK = 8
PFNET_HIDDEN = 64
VOCAB_SIZE = 256
PATCH_SIZE = 4


# =============================================================================
# Componentes compartidos
# =============================================================================

def _ste_round(x: Tensor) -> Tensor:
    """Round con Straight-Through Estimator: forward redondea, backward pasa directo."""
    return x + (x.round() - x).detach()


class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization."""

    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: Tensor) -> Tensor:
        norm = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * norm * self.weight


class BitLinear(nn.Module):
    """Linear con cuantización ternaria {-1, 0, 1} y STE."""

    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.empty(out_features, in_features))
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.weight, std=0.02)

    def forward(self, x: Tensor) -> Tensor:
        w = self.weight
        w_scale = w.abs().mean()
        w_normalized = w / (w_scale + 1e-8)
        w_ternary = torch.clamp(_ste_round(w_normalized), -1.0, 1.0)
        x_scale = x.abs().amax(dim=-1, keepdim=True)
        x_normalized = x / (x_scale + 1e-8)
        x_quant = torch.clamp(_ste_round(x_normalized * 127.0), -128.0, 127.0)
        output = F.linear(x_quant, w_ternary)
        output = output * (x_scale * w_scale / 127.0)
        return output


class FPResidual(nn.Module):
    """Corrección FP16 de rango bajo en paralelo con BitLinear."""

    def __init__(self, in_dim: int, out_dim: int | None = None, rank: int = FP_RANK):
        super().__init__()
        out_dim = out_dim or in_dim
        self.fp_A = nn.Linear(in_dim, rank, bias=False)
        self.fp_B = nn.Linear(rank, out_dim, bias=False)
        self._init_weights()

    def _init_weights(self):
        nn.init.normal_(self.fp_A.weight, std=0.02)
        nn.init.zeros_(self.fp_B.weight)

    def forward(self, x: Tensor) -> Tensor:
        return self.fp_B(F.silu(self.fp_A(x)))


class SwiGLU(nn.Module):
    """FFN con gating multiplicativo SwiGLU."""

    def __init__(self, d_model: int = D_MODEL, d_ff: int = D_FF):
        super().__init__()
        self.gate_proj = BitLinear(d_model, d_ff)
        self.up_proj = BitLinear(d_model, d_ff)
        self.down_proj = BitLinear(d_ff, d_model)
        self.down_fp_residual = FPResidual(d_ff, d_model)

    def forward(self, x: Tensor) -> Tensor:
        gate = F.silu(self.gate_proj(x))
        up = self.up_proj(x)
        hidden = gate * up
        return self.down_proj(hidden) + self.down_fp_residual(hidden)


# Memory budget for PFNet:
# Per layer:  (1024×64 + 64×1024) × 1 bit = 16 KB
# 14 layers:  14 × 16 KB = 224 KB  → fits in L2 (~256–512 KB per core)
# Main model: ~44 MB                → streams from RAM layer by layer
# Recurrent state: 14 × 1024 × 2B = 28 KB → fits in L2 alongside PFNet


class PFNet(nn.Module):
    """
    Prefetch-Aware Nonlinear Residual.

    A tiny 2-layer ternary MLP that runs in parallel with the main
    MLGRU block. Sized so all PFNet weights across all layers fit in
    CPU L2 cache (~224 KB for 14 layers at d_model=1024, hidden=64),
    allowing them to remain cache-resident while main weights stream
    from RAM during inference.

    Output is added as a residual to the block output:
        block_out = mlgru_out + ffn_out + pfnet_out

    Weights are ternary (BitLinear), activations are SiLU.
    No FPResidual correction branch — kept intentionally minimal.
    """

    def __init__(self, d_model: int, hidden: int = 64):
        super().__init__()
        self.w1 = BitLinear(d_model, hidden)
        self.w2 = BitLinear(hidden, d_model)
        self.norm = RMSNorm(d_model)

    def forward(self, x: Tensor) -> Tensor:
        return self.w2(F.silu(self.w1(self.norm(x))))


# =============================================================================
# 1. BitEmbedding — Embedding cuantizado a ternario (Slender-Mamba)
# =============================================================================

class BitEmbedding(nn.Module):
    """
    Capa de embedding cuantizada a 1.58 bits (ternario).

    Basado en Slender-Mamba (COLING 2025).
    Cuantiza la weight matrix del embedding a {-1, 0, 1} con un scale FP16
    por fila. En lookup: embedding = ternary_row * scale_row.

    Para byte-level (vocab_size=256) la tabla es pequena (256 * embed_dim),
    pero mantener todo ternario de pies a cabeza es el principio clave.
    """

    def __init__(self, vocab_size: int, embed_dim: int):
        super().__init__()
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.weight = nn.Parameter(torch.randn(vocab_size, embed_dim) * 0.02)
        self.scale = nn.Parameter(torch.ones(vocab_size, 1))

    def forward(self, indices: Tensor) -> Tensor:
        """
        Args:
            indices: cualquier shape de indices (ej: [B, S, 4] para patches)
        Returns:
            [..., embed_dim] FP16/FP32
        """
        # Cuantizar pesos en forward (STE para backward)
        w_scale = self.weight.abs().mean(dim=1, keepdim=True)  # [vocab, 1]
        w_ternary = (self.weight / (w_scale + 1e-8)).round().clamp(-1, 1)
        w_ternary = w_ternary.detach() + self.weight - self.weight.detach()  # STE

        # Lookup ternario * scale
        return F.embedding(indices, w_ternary * self.scale)


# =============================================================================
# 2. BytePatchEmbeddingV2 — con BitEmbedding en vez de nn.Embedding
# =============================================================================

class BytePatchEmbeddingV2(nn.Module):
    """
    Convierte patches de 4 bytes en embeddings de d_model dims.
    Usa BitEmbedding (ternario) en vez de nn.Embedding (FP16).

    4 bytes * 8 dims = 32 features -> BitLinear(32, d_model).
    """

    def __init__(self, d_model: int = D_MODEL, patch_size: int = PATCH_SIZE,
                 bytes_per_embed: int = 8):
        super().__init__()
        self.patch_size = patch_size
        self.bytes_per_embed = bytes_per_embed
        self.byte_embed = BitEmbedding(256, bytes_per_embed)
        self.patch_proj = BitLinear(patch_size * bytes_per_embed, d_model)

    def forward(self, patches: Tensor) -> Tensor:
        """patches: [B, S, 4] uint8/long -> [B, S, d_model]"""
        B, S, P = patches.shape
        embeds = self.byte_embed(patches.long())           # [B, S, 4, 8]
        embeds = embeds.reshape(B, S, P * self.bytes_per_embed)  # [B, S, 32]
        return self.patch_proj(embeds)                      # [B, S, d_model]


# =============================================================================
# 3. MatMulFreeTokenMixer — MLGRU sin multiplicacion de matrices
# =============================================================================

class MatMulFreeTokenMixer(nn.Module):
    """
    Reemplaza SlidingWindowAttention con un token mixer sin MatMul.

    Basado en "Scalable MatMul-free Language Modeling" (Zhu et al., 2024).
    Usa un GRU modificado (MLGRU) que solo usa productos elemento a elemento
    (Hadamard products) en vez de Q@K^T.

    Estructura por timestep:
        f_t = sigmoid(BitLinear_f(x_t))      # forget gate
        c_t = silu(BitLinear_c(x_t))         # candidate
        g_t = sigmoid(BitLinear_g(x_t))      # output gate
        h_t = f_t * h_{t-1} + (1 - f_t) * c_t   # Hadamard, no MatMul
        o_t = g_t * h_t

    Ventajas sobre atencion en CPU:
    - O(d_model) ops por token vs O(window_size * d_model)
    - Estado h: [d_model] = 4KB en FP16 -> siempre en L1
    - Zero KV cache: sin crecimiento de memoria con contexto largo
    """

    def __init__(self, d_model: int, fp_rank: int = FP_RANK):
        super().__init__()
        self.d_model = d_model

        # 3 proyecciones con BitLinear ternario
        self.forget_proj = BitLinear(d_model, d_model)
        self.cand_proj = BitLinear(d_model, d_model)
        self.output_proj = BitLinear(d_model, d_model)

        # FPResidual paralelo en output
        self.fp_residual = FPResidual(d_model, d_model, rank=fp_rank)

        # Estado inicial (buffer, no parameter)
        self.register_buffer('h_init', torch.zeros(d_model))

    def forward(
        self,
        x: Tensor,                    # [B, S, d_model]
        h: Tensor | None = None       # [B, d_model] estado previo
    ) -> tuple[Tensor, Tensor]:
        """
        Training: procesar secuencia completa con loop temporal.
        Inferencia: procesar un token con estado explicito.

        Returns: (output [B, S, d_model], h_final [B, d_model])
        """
        B, S, D = x.shape

        if h is None:
            h = self.h_init.unsqueeze(0).expand(B, -1).clone()

        # Proyecciones ternarias
        f = torch.sigmoid(self.forget_proj(x))    # [B, S, D]
        c = F.silu(self.cand_proj(x))             # [B, S, D]
        g = torch.sigmoid(self.output_proj(x))    # [B, S, D]

        # Scan temporal: h_t = f_t * h_{t-1} + (1 - f_t) * c_t
        outputs = []
        for t in range(S):
            h = f[:, t] * h + (1 - f[:, t]) * c[:, t]
            outputs.append(g[:, t] * h)

        out = torch.stack(outputs, dim=1)  # [B, S, D]

        # FPResidual en paralelo (lee x, no out)
        out = out + self.fp_residual(x)

        return out, h

    def forward_one_token(
        self,
        x: Tensor,   # [B, D]
        h: Tensor     # [B, D]
    ) -> tuple[Tensor, Tensor]:
        """
        Inferencia recurrente: O(D) ops, sin KV cache.
        """
        # Proyecciones para un solo vector (unsqueeze/squeeze para BitLinear)
        x_3d = x.unsqueeze(1)  # [B, 1, D]

        f = torch.sigmoid(self.forget_proj(x_3d)).squeeze(1)   # [B, D]
        c = F.silu(self.cand_proj(x_3d)).squeeze(1)            # [B, D]
        g = torch.sigmoid(self.output_proj(x_3d)).squeeze(1)   # [B, D]

        h_new = f * h + (1 - f) * c
        o = g * h_new

        fp_out = self.fp_residual(x_3d).squeeze(1)  # [B, D]
        return o + fp_out, h_new


# =============================================================================
# 4. BitHead — Prediction head ternario (Slender-Mamba)
# =============================================================================

class BitHead(nn.Module):
    """
    Prediction head ternario para byte prediction.

    Slender-Mamba cuantiza el lm_head con BitLinear.
    Para vocab_size=256 (byte-level) la degradacion es minima.

    Original:  nn.Linear(2048, 256) FP16 -> 4 heads * 1MB = 4MB
    BitHead:   BitLinear(2048, 256) ternario -> 4 heads * 0.125MB = 0.5MB
    """

    def __init__(self, d_model: int, vocab_size: int = VOCAB_SIZE):
        super().__init__()
        self.proj = BitLinear(d_model, vocab_size)
        self.output_scale = nn.Parameter(torch.ones(1))

    def forward(self, x: Tensor) -> Tensor:
        """x: [B, S, d_model] -> [B, S, 256]"""
        return self.proj(x) * self.output_scale


# =============================================================================
# 5. CPU1BlockV2 — Bloque con MLGRU + SwiGLU, todo ternario
# =============================================================================

class CPU1BlockV2(nn.Module):
    """
    Bloque actualizado: MLGRU + SwiGLU + PFNet.

    Perfil de memoria en inferencia por bloque:
        Original (atencion): KV cache 516 * 2048 * 2 * 2 bytes = 4MB/bloque
        MLGRU:               Estado h = 1024 * 2 bytes = 2KB/bloque
        PFNet:               16 KB/bloque (cache-resident in L2)
    """

    def __init__(self, d_model: int = D_MODEL, d_ff: int = D_FF,
                 fp_rank: int = FP_RANK, pfnet_hidden: int = PFNET_HIDDEN):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.norm2 = RMSNorm(d_model)
        self.token_mixer = MatMulFreeTokenMixer(d_model, fp_rank)
        self.ffn = SwiGLU(d_model, d_ff)
        self.pfnet = PFNet(d_model, hidden=pfnet_hidden)

    def forward(
        self,
        x: Tensor,                    # [B, S, d_model]
        h: Tensor | None = None       # [B, d_model]
    ) -> tuple[Tensor, Tensor]:
        # Pre-norm -> MLGRU -> residual
        residual = x
        mixer_out, h_new = self.token_mixer(self.norm1(x), h)
        x = residual + mixer_out
        # Pre-norm -> FFN -> residual + PFNet nonlinear residual
        residual = x
        x = self.ffn(self.norm2(x))
        x = residual + x + self.pfnet(residual)
        return x, h_new

    def forward_one_token(
        self,
        x: Tensor,   # [B, D]
        h: Tensor     # [B, D]
    ) -> tuple[Tensor, Tensor]:
        """Inferencia de 1 token a traves del bloque."""
        residual = x
        x_3d = x.unsqueeze(1)  # [B, 1, D]
        normed = self.norm1(x_3d).squeeze(1)  # [B, D]
        mixer_out, h_new = self.token_mixer.forward_one_token(normed, h)
        x = residual + mixer_out  # residual
        residual = x
        x_3d = x.unsqueeze(1)
        pfnet_out = self.pfnet(x_3d).squeeze(1)  # PFNet from pre-norm input
        x = residual + self.ffn(self.norm2(x_3d)).squeeze(1) + pfnet_out
        return x, h_new


# =============================================================================
# 6. CPU1ModelV2 — Modelo completo actualizado
# =============================================================================

class CPU1ModelV2(nn.Module):
    """
    CPU-1 v2: todo ternario, sin MatMul en token mixing, con PFNet.

    Componentes cambiados vs v1:
        BytePatchEmbedding     -> BytePatchEmbeddingV2 (BitEmbedding)
        SlidingWindowAttention -> MatMulFreeTokenMixer (MLGRU)
        nn.Linear heads        -> BitHead (ternario)
        RoPE                   -> eliminado
        PFNet                  -> nonlinear residual branch per block

    Estado de inferencia:
        MLGRU state: 14 * 1024 * 2 bytes = 28KB total -> en L2
        PFNet weights: 14 * 16KB = 224KB -> en L2
        (antes: 20 * 516 * 2048 * 2 * 2 bytes = 80MB -> no cabia en L3)
    """

    def __init__(self, config: dict | None = None):
        super().__init__()
        config = config or {}
        d_model = config.get('d_model', D_MODEL)
        n_layers = config.get('n_layers', N_LAYERS)
        d_ff = config.get('d_ff', D_FF)
        fp_rank = config.get('fp_rank', FP_RANK)
        pfnet_hidden = config.get('pfnet_hidden', PFNET_HIDDEN)

        self.d_model = d_model
        self.n_layers = n_layers

        self.patch_embedding = BytePatchEmbeddingV2(d_model)
        self.blocks = nn.ModuleList([
            CPU1BlockV2(d_model, d_ff, fp_rank, pfnet_hidden) for _ in range(n_layers)
        ])
        self.norm = RMSNorm(d_model)

        # 4 heads ternarios (Slender-Mamba style)
        self.byte_heads = nn.ModuleList([
            BitHead(d_model, VOCAB_SIZE) for _ in range(4)
        ])

        self.use_gradient_checkpointing = config.get(
            'use_gradient_checkpointing', False
        )

    def forward(
        self,
        patches: Tensor,                     # [B, S, 4] uint8/long
        hidden_states: list | None = None    # lista de n_layers tensores [B, d_model]
    ) -> tuple[tuple, list]:
        """
        Returns:
            logits: tuple de 4 tensores [B, S, 256]
            hidden_states: lista de n_layers tensores [B, d_model]
        """
        x = self.patch_embedding(patches)

        if hidden_states is None:
            hidden_states = [None] * len(self.blocks)

        new_states = []

        if self.use_gradient_checkpointing and self.training:
            from torch.utils.checkpoint import checkpoint
            for i, block in enumerate(self.blocks):
                def _block_fn(x_in, h_in, _block=block):
                    return _block(x_in, h_in)
                x, h_new = checkpoint(_block_fn, x, hidden_states[i],
                                      use_reentrant=False)
                new_states.append(h_new)
        else:
            for i, block in enumerate(self.blocks):
                # Prefetch pesos de capa i+2 mientras procesamos capa i
                if i + 2 < len(self.blocks):
                    _ = self.blocks[i + 2].token_mixer.forget_proj.weight.data_ptr()

                x, h_new = block(x, hidden_states[i])
                new_states.append(h_new)

        x = self.norm(x)
        logits = tuple(head(x) for head in self.byte_heads)

        return logits, new_states

    def generate_bytes(
        self,
        prompt_bytes: bytes,
        max_tokens: int = 200,
        temperature: float = 1.0,
        top_p: float = 0.9,
    ) -> bytes:
        """
        Inferencia recurrente token por token.
        Cero KV cache. Estado MLGRU de 28KB fijo.
        """
        self.eval()
        device = next(self.parameters()).device

        # Convertir prompt a patches
        patches = bytes_to_patches(prompt_bytes).to(device)
        hidden_states = None

        # Prefill: procesar prompt completo
        with torch.no_grad():
            _, hidden_states = self.forward(patches, hidden_states)

        # Generacion autoregresiva
        generated = []
        last_patch = patches[:, -1:, :]

        for _ in range(max_tokens // PATCH_SIZE):
            with torch.no_grad():
                logits, hidden_states = self.forward(last_patch, hidden_states)

            # Samplear los 4 bytes del proximo patch
            next_bytes = []
            for head_logits in logits:
                probs = F.softmax(head_logits[0, 0] / temperature, dim=-1)
                # Top-p (nucleus) sampling
                sorted_probs, sorted_idx = probs.sort(descending=True)
                cumsum = sorted_probs.cumsum(dim=0)
                mask = cumsum - sorted_probs > top_p
                sorted_probs[mask] = 0.0
                sorted_probs /= sorted_probs.sum()
                byte_val = sorted_idx[torch.multinomial(sorted_probs, 1)].item()
                next_bytes.append(byte_val)

            generated.extend(next_bytes)
            last_patch = torch.tensor(
                next_bytes, dtype=torch.long, device=device
            ).reshape(1, 1, PATCH_SIZE)

        return bytes(generated)

    def generate_one_patch(
        self,
        x_patch: Tensor,        # [B, D]
        hidden_states: list,     # lista de n_layers [B, D]
    ) -> tuple[Tensor, list]:
        """
        Procesar 1 patch a traves de TODAS las capas.
        Maximiza reutilizacion del estado h en cache (28KB en L2).
        """
        new_states = []
        for i, block in enumerate(self.blocks):
            x_patch, h_new = block.forward_one_token(x_patch, hidden_states[i])
            new_states.append(h_new)
        return x_patch, new_states

    def prepare_for_inference(self):
        """
        Preparar pesos para inferencia CPU:
        1. Hacer pesos contiguos
        2. Transponer pesos para acceso secuencial en vector-matrix multiply
        3. Verificar alineamiento de cache line
        """
        for block in self.blocks:
            for name, param in block.named_parameters():
                if not param.data.is_contiguous():
                    param.data = param.data.contiguous()

        # Transponer pesos de BitLinear para inferencia eficiente
        _prepare_bitlinear_weights(self)

        return True


# =============================================================================
# 7. Utilidades de conversion bytes <-> patches
# =============================================================================

def bytes_to_patches(raw: bytes) -> Tensor:
    """
    Convierte bytes crudos a patches de 4 bytes.

    Args:
        raw: bytes de entrada
    Returns:
        [1, n_patches, 4] long
    """
    n = len(raw)
    pad = (PATCH_SIZE - n % PATCH_SIZE) % PATCH_SIZE
    raw = raw + b'\x00' * pad

    n_patches = len(raw) // PATCH_SIZE
    patches = []
    for i in range(n_patches):
        chunk = raw[i * PATCH_SIZE:(i + 1) * PATCH_SIZE]
        patches.append(list(chunk))

    return torch.tensor(patches, dtype=torch.long).unsqueeze(0)


# =============================================================================
# 8. Optimizaciones CPU para inferencia
# =============================================================================

def _prepare_bitlinear_weights(model: nn.Module):
    """
    Transponer y hacer contigua la matriz de pesos de todas las capas
    BitLinear ANTES de inferencia.

    En inferencia (1 token): y = x @ W.T
    Si guardamos W_T = W.T.contiguous(), el acceso es secuencial en memoria.
    Ganancia: hasta 2x en throughput de memory bandwidth para matmul 1xNxM.
    """
    for module in model.modules():
        if isinstance(module, BitLinear):
            with torch.no_grad():
                module.weight_T = module.weight.data.T.contiguous()


def assert_aligned(tensor: Tensor, alignment: int = 64):
    """Verifica que un tensor esta alineado a cache line (64 bytes en x86)."""
    offset = tensor.data_ptr() % alignment
    assert offset == 0, f"Tensor no alineado: offset={offset}"


def pack_ternary_to_int8(weight: Tensor) -> tuple[Tensor, Tensor]:
    """
    Empaqueta pesos {-1, 0, 1} en INT8 para inferencia CPU eficiente.

    PyTorch soporta matmul INT8 via torch._int_mm.
    Ventajas vs FP16:
    - 2x menos bandwidth de memoria
    - INT8 matmul en AVX2: 32 elementos/ciclo vs 8 para FP32

    Returns:
        w_int8:  pesos en INT8 [-1, 0, 1]
        w_scale: escalar de re-escalado FP32
    """
    w_scale = weight.abs().mean()
    w_ternary = (weight / (w_scale + 1e-8)).round().clamp(-1, 1)
    w_int8 = w_ternary.to(torch.int8)
    return w_int8, w_scale


def ternary_matmul_int8(x: Tensor, w_int8: Tensor, scale: float) -> Tensor:
    """
    Matmul eficiente usando INT8 acumulacion.

    Args:
        x: [1, d_in] FP32/FP16
        w_int8: [d_out, d_in] INT8
        scale: factor de escala FP32
    Returns:
        [1, d_out] FP32
    """
    x_scale = x.abs().max() / 127.0
    x_int8 = (x / (x_scale + 1e-8)).round().clamp(-128, 127).to(torch.int8)

    # INT8 matmul (requiere 2D tensors y padding a multiplo de 8 para _int_mm)
    m, k = x_int8.shape
    k2 = w_int8.shape[1]
    assert k == k2, f"Dimension mismatch: x={k}, w={k2}"

    # Pad k a multiplo de 8 si es necesario para _int_mm
    pad_k = (8 - k % 8) % 8
    if pad_k > 0:
        x_int8 = F.pad(x_int8, (0, pad_k))
        w_int8 = F.pad(w_int8, (0, pad_k))

    out = torch._int_mm(x_int8, w_int8.T).to(torch.float32)
    return out * (x_scale * scale)


def pack_model_for_int8_inference(model: CPU1ModelV2) -> dict:
    """
    Empaqueta todos los pesos BitLinear del modelo a INT8.
    Retorna un dict con los pesos empaquetados para inferencia.
    """
    packed = {}
    for name, module in model.named_modules():
        if isinstance(module, BitLinear):
            w_int8, w_scale = pack_ternary_to_int8(module.weight.data)
            packed[name] = {'w_int8': w_int8, 'w_scale': w_scale}
    return packed


def compile_for_inference(model: CPU1ModelV2, aggressive: bool = True):
    """
    Aplicar torch.compile al modelo para inferencia.

    torch.compile fusiona ops pequenas (sigmoid, mul, add) en un solo kernel.
    Especialmente efectivo para el MLGRU con muchos Hadamard consecutivos.

    Args:
        model: modelo a compilar
        aggressive: True = max-autotune (lento en compilar, rapido en runtime)
    Returns:
        modelo compilado
    """
    mode = "max-autotune" if aggressive else "reduce-overhead"
    try:
        compiled = torch.compile(model, mode=mode, backend="inductor")
        return compiled
    except Exception:
        # Fallback si torch.compile no esta disponible o falla
        return model


# =============================================================================
# 9. Ablation configs y benchmarks
# =============================================================================

def run_architecture_ablation():
    """
    Tabla de ablation para el paper.
    4 variantes a medir en el MISMO hardware CPU consumer.

    Hipotesis:
    - V3 > V2 en tokens/seg (MLGRU elimina KV cache, evita RAM stalls)
    - V4 > V3 si byte-level tiene menor perplexity por byte vs BPE
    - V4 > V1 porque byte-level + MLGRU > BPE + ternario Mamba
    """
    configs = {
        'v1_slender_mamba_bpe': {
            'arch': 'mamba2',
            'tokenizer': 'bpe',
            'quantization': 'ternary_full',
            'note': 'baseline Slender-Mamba style',
        },
        'v2_cpu1_attention': {
            'arch': 'transformer_sliding_window',
            'tokenizer': 'byte',
            'quantization': 'ternary_partial',
            'note': 'CPU-1 original del README',
        },
        'v3_cpu1_mlgru': {
            'arch': 'mlgru_matmulfree',
            'tokenizer': 'byte',
            'quantization': 'ternary_partial',
            'note': 'MLGRU reemplaza atencion',
        },
        'v4_cpu1_mlgru_full': {
            'arch': 'mlgru_matmulfree',
            'tokenizer': 'byte',
            'quantization': 'ternary_full',
            'note': 'todo ternario, sin MatMul',
        },
        'v5_cpu1_170m_pfnet': {
            'arch': 'mlgru_matmulfree_pfnet',
            'tokenizer': 'byte',
            'quantization': 'ternary_full',
            'note': '170M, PFNet nonlinear residual',
        },
    }

    metrics = [
        'tokens_per_sec_cpu',
        'latency_p50_ms',
        'ram_mb',
        'model_size_mb',
        'perplexity_fineweb',
        'cache_miss_rate',
        'state_size_kb',
    ]

    return configs, metrics


def benchmark_cpu_inference(
    model: CPU1ModelV2,
    n_patches: int = 100,
    warmup: int = 10,
) -> dict:
    """
    Benchmark de inferencia CPU: mide throughput y latencia.

    Args:
        model: modelo v2
        n_patches: cantidad de patches a generar
        warmup: patches de warmup (no contados)
    Returns:
        dict con metricas
    """
    model.eval()
    model.prepare_for_inference()
    device = next(model.parameters()).device

    # Inicializar estado
    hidden_states = [None] * len(model.blocks)
    dummy_patch = torch.randint(0, 256, (1, 1, PATCH_SIZE), device=device)

    # Warmup
    with torch.no_grad():
        for _ in range(warmup):
            logits, hidden_states = model.forward(dummy_patch, hidden_states)

    # Benchmark
    latencies = []
    with torch.no_grad():
        for _ in range(n_patches):
            t0 = time.perf_counter()
            logits, hidden_states = model.forward(dummy_patch, hidden_states)
            t1 = time.perf_counter()
            latencies.append((t1 - t0) * 1000)  # ms

    latencies.sort()
    total_time_s = sum(latencies) / 1000.0
    bytes_generated = n_patches * PATCH_SIZE

    return {
        'patches_per_sec': n_patches / total_time_s,
        'bytes_per_sec': bytes_generated / total_time_s,
        'latency_p50_ms': latencies[len(latencies) // 2],
        'latency_p99_ms': latencies[int(len(latencies) * 0.99)],
        'total_time_s': total_time_s,
        'n_patches': n_patches,
    }


def benchmark_byte_vs_bpe_throughput():
    """
    Test critico: patches de bytes vs tokens BPE en CPU.

    Hipotesis: SI, porque:
    1. vocab_size=256 vs 32K -> embedding lookup 128x mas pequeno
    2. Tokenizador BPE tiene overhead de Python/Rust
    3. Patches de 4 bytes tienen longitud fija -> sin padding

    Contraargumento: secuencia en bytes es 3-5x mas larga que en tokens BPE.

    Placeholder: requiere modelo BPE baseline para comparacion directa.
    """
    pass


# =============================================================================
# 10. Generador optimizado con pre-alocacion de buffers
# =============================================================================

class CPU1GeneratorV2:
    """
    Generador optimizado para inferencia CPU.
    Pre-aloca buffers para evitar garbage collector jitter.
    """

    def __init__(self, model: CPU1ModelV2):
        self.model = model
        self.model.eval()
        self.device = next(model.parameters()).device

    def generate(
        self,
        prompt: str,
        max_bytes: int = 200,
        temperature: float = 0.8,
        top_p: float = 0.9,
    ) -> str:
        """Genera texto desde un prompt string."""
        raw = prompt.encode('utf-8')
        generated = self.model.generate_bytes(
            raw, max_tokens=max_bytes,
            temperature=temperature, top_p=top_p,
        )
        # Decodificar, removiendo trailing nulls
        return generated.rstrip(b'\x00').decode('utf-8', errors='replace')

    @torch.no_grad()
    def generate_optimized(
        self,
        prompt_bytes: bytes,
        max_patches: int = 50,
        temperature: float = 0.8,
        top_p: float = 0.9,
    ) -> bytes:
        """
        Generacion optimizada: pre-aloca buffers y usa forward_one_token.

        Pre-aloca todos los buffers UNA VEZ antes del loop para evitar
        allocaciones que disparan el garbage collector.
        """
        # Prefill
        patches = bytes_to_patches(prompt_bytes).to(self.device)
        _, hidden_states = self.model.forward(patches, None)

        # Pre-alocar buffers
        output_buf = bytearray(max_patches * PATCH_SIZE)
        probs_buf = torch.zeros(VOCAB_SIZE, device=self.device)

        # Embedding del ultimo patch para empezar generacion
        last_patch = patches[:, -1:, :]
        x = self.model.patch_embedding(last_patch).squeeze(1)  # [B, D]

        for i in range(max_patches):
            # Forward a traves de todas las capas
            x_out, hidden_states = self.model.generate_one_patch(x, hidden_states)

            # Norm final y heads
            x_normed = self.model.norm(x_out.unsqueeze(1))  # [B, 1, D]
            next_bytes = []
            for head in self.model.byte_heads:
                logits = head(x_normed).squeeze(1).squeeze(0)  # [256]
                torch.softmax(logits / temperature, dim=-1, out=probs_buf)

                # Top-p filtering in-place
                sorted_probs, sorted_idx = probs_buf.sort(descending=True)
                cumsum = sorted_probs.cumsum(dim=0)
                mask = cumsum - sorted_probs > top_p
                sorted_probs[mask] = 0.0
                sorted_probs /= sorted_probs.sum()

                byte_val = sorted_idx[torch.multinomial(sorted_probs, 1)].item()
                next_bytes.append(byte_val)

            for j, bv in enumerate(next_bytes):
                output_buf[i * PATCH_SIZE + j] = bv

            # Embed siguiente patch
            next_patch = torch.tensor(
                next_bytes, dtype=torch.long, device=self.device
            ).reshape(1, 1, PATCH_SIZE)
            x = self.model.patch_embedding(next_patch).squeeze(1)

        return bytes(output_buf)


# =============================================================================
# 11. Tests
# =============================================================================

def count_parameters(model: nn.Module) -> int:
    """Cuenta total de parametros del modelo."""
    return sum(p.numel() for p in model.parameters())


def print_parameter_breakdown(model: CPU1ModelV2):
    """Prints a detailed parameter breakdown: embedding, per-layer (MLGRU + FFN + PFNet), heads, total."""
    total = count_parameters(model)
    embed_params = count_parameters(model.patch_embedding)
    norm_params = count_parameters(model.norm)
    head_params = sum(count_parameters(h) for h in model.byte_heads)
    block_params = count_parameters(model.blocks)

    print(f"Parameter breakdown (~{total / 1e6:.0f}M total):")
    print(f"  Embedding:       {embed_params:>12,}")
    print(f"  Blocks (x{model.n_layers}):    {block_params:>12,}")

    block = model.blocks[0]
    mixer_p = count_parameters(block.token_mixer)
    ffn_p = count_parameters(block.ffn)
    pfnet_p = count_parameters(block.pfnet)
    norms_p = count_parameters(block.norm1) + count_parameters(block.norm2)
    print(f"    Per block:")
    print(f"      MLGRU mixer: {mixer_p:>10,}")
    print(f"      SwiGLU FFN:  {ffn_p:>10,}")
    print(f"      PFNet:       {pfnet_p:>10,}")
    print(f"      Norms:       {norms_p:>10,}")

    print(f"  Final norm:      {norm_params:>12,}")
    print(f"  Heads (x4):      {head_params:>12,}")
    print(f"  Total:           {total:>12,}")


def test_bit_embedding():
    """Verifica BitEmbedding."""
    emb = BitEmbedding(256, 8)
    indices = torch.randint(0, 256, (2, 16, 4))
    out = emb(indices)
    assert out.shape == (2, 16, 4, 8), f"Shape error: {out.shape}"
    print("[OK] BitEmbedding OK")


def test_byte_patch_embedding_v2():
    """Verifica BytePatchEmbeddingV2."""
    emb = BytePatchEmbeddingV2(d_model=128)
    patches = torch.randint(0, 256, (2, 16, 4))
    out = emb(patches)
    assert out.shape == (2, 16, 128), f"Shape error: {out.shape}"
    print("[OK] BytePatchEmbeddingV2 OK")


def test_matmul_free_token_mixer():
    """Verifica MatMulFreeTokenMixer forward y forward_one_token."""
    D = 128
    mixer = MatMulFreeTokenMixer(D, fp_rank=4)
    x = torch.randn(2, 8, D)

    # Forward completo
    out, h = mixer(x)
    assert out.shape == (2, 8, D), f"Shape error: {out.shape}"
    assert h.shape == (2, D), f"State shape error: {h.shape}"

    # Forward one token
    x1 = torch.randn(2, D)
    h1 = torch.zeros(2, D)
    out1, h1_new = mixer.forward_one_token(x1, h1)
    assert out1.shape == (2, D), f"Shape error: {out1.shape}"
    assert h1_new.shape == (2, D), f"State shape error: {h1_new.shape}"

    print("[OK] MatMulFreeTokenMixer OK")


def test_bit_head():
    """Verifica BitHead."""
    head = BitHead(128, 256)
    x = torch.randn(2, 8, 128)
    out = head(x)
    assert out.shape == (2, 8, 256), f"Shape error: {out.shape}"
    print("[OK] BitHead OK")


def test_cpu1_block_v2():
    """Verifica CPU1BlockV2 forward y forward_one_token."""
    D = 128
    block = CPU1BlockV2(d_model=D, d_ff=256, fp_rank=4)
    x = torch.randn(2, 8, D)

    # Forward completo
    out, h = block(x)
    assert out.shape == (2, 8, D), f"Shape error: {out.shape}"
    assert h.shape == (2, D), f"State shape error: {h.shape}"

    # Forward one token
    x1 = torch.randn(2, D)
    h1 = torch.zeros(2, D)
    out1, h1_new = block.forward_one_token(x1, h1)
    assert out1.shape == (2, D), f"Shape error: {out1.shape}"

    print("[OK] CPU1BlockV2 OK")


def test_cpu1_model_v2():
    """Verifica CPU1ModelV2 forward pass."""
    config = {'d_model': 128, 'n_layers': 2, 'd_ff': 256, 'fp_rank': 4}
    model = CPU1ModelV2(config)
    model.eval()

    patches = torch.randint(0, 256, (2, 8, PATCH_SIZE))

    with torch.no_grad():
        logits, states = model(patches)

    assert len(logits) == 4, f"Expected 4 heads, got {len(logits)}"
    for i, l in enumerate(logits):
        assert l.shape == (2, 8, VOCAB_SIZE), (
            f"Head {i}: shape {l.shape}, expected (2, 8, {VOCAB_SIZE})"
        )
    assert len(states) == 2, f"Expected 2 states, got {len(states)}"

    print(f"[OK] CPU1ModelV2 forward OK: 4 x {logits[0].shape}")


def test_generate_bytes():
    """Verifica generacion de bytes."""
    config = {'d_model': 128, 'n_layers': 2, 'd_ff': 256, 'fp_rank': 4}
    model = CPU1ModelV2(config)

    generated = model.generate_bytes(b"Hello", max_tokens=16, temperature=1.0)
    assert len(generated) == 16, f"Expected 16 bytes, got {len(generated)}"
    assert all(0 <= b <= 255 for b in generated), "Invalid byte values"

    print(f"[OK] generate_bytes OK: {len(generated)} bytes generados")


def test_parameter_count_v2():
    """Verifica parametros del modelo completo v2 (~170M)."""
    model = CPU1ModelV2()
    total = count_parameters(model)
    print_parameter_breakdown(model)
    print(f"\n[OK] Test de parametros v2 paso: {total:,}")


def test_prepare_for_inference():
    """Verifica que prepare_for_inference funciona."""
    config = {'d_model': 128, 'n_layers': 2, 'd_ff': 256, 'fp_rank': 4}
    model = CPU1ModelV2(config)
    result = model.prepare_for_inference()
    assert result is True

    # Verificar que los pesos transpuestos existen
    for module in model.modules():
        if isinstance(module, BitLinear):
            assert hasattr(module, 'weight_T'), "weight_T not created"
            assert module.weight_T.shape == module.weight.T.shape

    print("[OK] prepare_for_inference OK")


def test_pack_ternary():
    """Verifica empaquetado a INT8."""
    w = torch.randn(256, 128)
    w_int8, w_scale = pack_ternary_to_int8(w)
    assert w_int8.dtype == torch.int8
    assert w_int8.shape == (256, 128)
    assert (w_int8.abs() <= 1).all(), "Valores fuera de {-1, 0, 1}"
    print("[OK] pack_ternary_to_int8 OK")


def test_bytes_to_patches():
    """Verifica conversion bytes -> patches."""
    raw = b"Hello World!"
    patches = bytes_to_patches(raw)
    assert patches.shape == (1, 3, 4), f"Shape error: {patches.shape}"
    assert patches[0, 0, 0].item() == ord('H')
    assert patches[0, 0, 1].item() == ord('e')
    print("[OK] bytes_to_patches OK")


def test_cpu1_block_v2_one_token_consistency():
    """
    Verifica que forward_one_token produce el mismo resultado
    que forward con S=1 para un bloque.
    """
    D = 128
    block = CPU1BlockV2(d_model=D, d_ff=256, fp_rank=4)
    block.eval()

    x = torch.randn(1, D)
    h = torch.randn(1, D)

    with torch.no_grad():
        # Via forward (3D)
        x_3d = x.unsqueeze(1)
        out_3d, h_3d = block(x_3d, h)
        out_3d = out_3d.squeeze(1)

        # Via forward_one_token (2D)
        out_2d, h_2d = block.forward_one_token(x, h)

    assert torch.allclose(out_3d, out_2d, atol=1e-5), (
        f"Outputs differ: max diff = {(out_3d - out_2d).abs().max()}"
    )
    assert torch.allclose(h_3d, h_2d, atol=1e-5), (
        f"States differ: max diff = {(h_3d - h_2d).abs().max()}"
    )

    print("[OK] forward_one_token consistency OK")


def test_benchmark():
    """Verifica que el benchmark funciona."""
    config = {'d_model': 128, 'n_layers': 2, 'd_ff': 256, 'fp_rank': 4}
    model = CPU1ModelV2(config)

    results = benchmark_cpu_inference(model, n_patches=20, warmup=5)
    assert results['patches_per_sec'] > 0
    assert results['latency_p50_ms'] > 0

    print(f"[OK] Benchmark: {results['bytes_per_sec']:.0f} bytes/sec, "
          f"p50={results['latency_p50_ms']:.1f}ms")


if __name__ == "__main__":
    print("=" * 60)
    print("Test 1: Componentes individuales v2")
    print("=" * 60)
    test_bit_embedding()
    test_byte_patch_embedding_v2()
    test_matmul_free_token_mixer()
    test_bit_head()
    test_cpu1_block_v2()

    print("\n" + "=" * 60)
    print("Test 2: Modelo completo v2")
    print("=" * 60)
    test_cpu1_model_v2()

    print("\n" + "=" * 60)
    print("Test 3: Generacion de bytes")
    print("=" * 60)
    test_generate_bytes()

    print("\n" + "=" * 60)
    print("Test 4: Consistencia forward_one_token")
    print("=" * 60)
    test_cpu1_block_v2_one_token_consistency()

    print("\n" + "=" * 60)
    print("Test 5: Inferencia CPU optimizations")
    print("=" * 60)
    test_prepare_for_inference()
    test_pack_ternary()

    print("\n" + "=" * 60)
    print("Test 6: Utilidades")
    print("=" * 60)
    test_bytes_to_patches()

    print("\n" + "=" * 60)
    print("Test 7: Benchmark")
    print("=" * 60)
    test_benchmark()

    print("\n" + "=" * 60)
    print("Test 8: Conteo de parametros (modelo completo)")
    print("=" * 60)
    test_parameter_count_v2()


In [ ]:
%%writefile /kaggle/working/train.py
"""
Training loop para CPU-1 con distillation loss.
Diseñado para Kaggle 2×T4 con DDP, compatible con single-GPU y CPU (dry-run).

Uso:
    # Dry-run (testing sin GPU)
    python train.py --dry-run --total-steps 10 --log-every 1

    # Single GPU con datos reales
    python train.py --data-path ./distillation_dataset --total-steps 100000

    # 2×T4 con DDP (Kaggle)
    torchrun --nproc_per_node=2 train.py --data-path ./distillation_dataset
"""

import os
import math
import json
import time
import contextlib
import argparse
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from torch import Tensor

from model_v2 import CPU1ModelV2, D_MODEL, VOCAB_SIZE, PATCH_SIZE


# =============================================================================
# 1. Distillation Loss
# =============================================================================

def distillation_loss(
    student_logits: tuple[Tensor, Tensor, Tensor, Tensor],  # 4 × [B, S, 256]
    target_bytes: Tensor,          # [B, S, 4] — ground truth next patch
    teacher_probs: Tensor | None,  # [B, S, 256] — distribucion marginalizada byte 0
    teacher_mask: Tensor,          # [B, S] bool — True donde hay senal teacher
    temperature: float = 4.0,
    alpha_kl: float = 0.9,
    alpha_ce: float = 0.1,
    compute_metrics: bool = False,
) -> tuple[Tensor, dict[str, float]]:
    """
    Loss combinada para distillation byte-level con teacher BPE.

    Args:
        student_logits: tupla de 4 tensores [B, S, 256], uno por byte del patch.
                        Son los logits crudos (pre-softmax) del student.
        target_bytes:   [B, S, 4] — bytes reales del proximo patch (ground truth)
        teacher_probs:  [B, S, 256] — distribucion de probabilidad del teacher
                        marginalizada al primer byte (viene de first-byte marginalization).
                        Sparse: la mayoria de los 256 valores son 0, solo ~5-20 tienen prob > 0.
        teacher_mask:   [B, S] bool — True en posiciones donde el teacher tiene senal
                        (donde un BPE boundary coincide con un patch boundary, ~20-30% del tiempo)
        temperature:    temperatura para suavizar distribuciones en KL (T=4.0 por default).
                        Mas alto = distribuciones mas planas = el student aprende mas de
                        las probabilidades "no top-1" del teacher (dark knowledge).
        alpha_kl:       peso del KL loss donde hay teacher (0.9 = 90% del loss viene del teacher)
        alpha_ce:       peso del CE loss donde hay teacher (0.1 = 10% hard target)
        compute_metrics: si True, calcula metricas detalladas. Fuerza GPU->CPU sync
                         (lento), usar solo cada N steps de logging.

    Returns:
        (loss, metrics): loss escalar diferenciable + dict de metricas (vacio si compute_metrics=False)

    Logica:
        - Head 0 (primer byte): KL(student/T, teacher/T) * T^2 donde hay senal + CE.
          El factor T^2 compensa que softmax(logits/T) produce gradientes T veces menores.
        - Heads 1-3 (bytes internos): solo CE contra ground truth.
          El teacher BPE no da info util sobre bytes internos a un token.
        - Loss final = promedio de las 4 cabezas.
    """
    metrics: dict[str, float] = {}

    # ===== Head 0: primer byte con posible señal teacher =====
    logits_0 = student_logits[0]  # [B, S, 256]
    targets_0 = target_bytes[:, :, 0].long()  # [B, S]

    ce_loss_0 = F.cross_entropy(
        logits_0.reshape(-1, 256),
        targets_0.reshape(-1),
        reduction='none',
    ).reshape_as(targets_0)  # [B, S]

    if teacher_probs is not None and teacher_mask.any():
        # Softmax con temperatura sobre logits del student: log(softmax(logits/T))
        student_log_probs_T = F.log_softmax(logits_0 / temperature, dim=-1)

        # Aplicar temperatura al teacher: como no tenemos los logits originales del teacher
        # (solo las probabilidades), usamos p^(1/T) que es equivalente a softmax(logits/T)
        # cuando p = softmax(logits). Despues renormalizamos.
        teacher_probs_T = teacher_probs ** (1.0 / temperature)
        teacher_probs_T = teacher_probs_T / (teacher_probs_T.sum(dim=-1, keepdim=True) + 1e-8)

        # KL divergence: mide cuanto difiere la distribucion del student de la del teacher.
        # sum(dim=-1) suma sobre los 256 bytes para obtener KL por posicion.
        kl = F.kl_div(
            student_log_probs_T,  # log-probs del student (log_target=False -> teacher en probs)
            teacher_probs_T,       # probs del teacher (no log)
            reduction='none',
            log_target=False,
        ).sum(dim=-1)  # [B, S] — un valor KL por posicion

        # Factor T^2: compensa que al dividir logits por T, los gradientes se escalan por 1/T.
        # Sin esto, el KL loss seria T veces mas debil de lo que deberia.
        kl_scaled = kl * (temperature ** 2)

        # Combinar con mascara: posiciones con teacher usan KL+CE mixta,
        # posiciones sin teacher usan solo CE contra ground truth
        mask_float = teacher_mask.float()
        loss_with_teacher = alpha_kl * kl_scaled + alpha_ce * ce_loss_0
        head0_loss = mask_float * loss_with_teacher + (1.0 - mask_float) * ce_loss_0

        if compute_metrics:
            n_teacher = mask_float.sum().clamp(min=1)
            metrics['loss_head0_kl'] = (kl_scaled * mask_float).sum().item() / n_teacher.item()
            metrics['teacher_coverage'] = mask_float.mean().item()
    else:
        head0_loss = ce_loss_0
        if compute_metrics:
            metrics['loss_head0_kl'] = 0.0
            metrics['teacher_coverage'] = 0.0

    total_loss = head0_loss.mean()

    if compute_metrics:
        metrics['loss_head0_ce'] = ce_loss_0.mean().item()

    # ===== Heads 1-3: solo CE, sin teacher =====
    heads_ce_sum = 0.0
    for i in range(1, 4):
        logits_i = student_logits[i]
        targets_i = target_bytes[:, :, i].long()
        ce_i = F.cross_entropy(
            logits_i.reshape(-1, 256),
            targets_i.reshape(-1),
            reduction='mean',
        )
        total_loss = total_loss + ce_i
        if compute_metrics:
            heads_ce_sum += ce_i.item()

    total_loss = total_loss / 4.0  # promedio sobre las 4 cabezas

    if compute_metrics:
        metrics['loss_heads_123_ce'] = heads_ce_sum / 3.0
        metrics['loss_total'] = total_loss.item()

    return total_loss, metrics


# =============================================================================
# 2. Dataset
# =============================================================================

class DistillationDataset(Dataset):
    """
    Carga patches pre-computados + teacher signals.

    Soporta:
    - HuggingFace Dataset / Parquet con document packing pre-hecho
    - Modo dry-run con datos aleatorios (para testing del loop)

    Cada sample:
        patches:       [seq_len, 4] long — input patches
        targets:       [seq_len, 4] long — next patches (shifted by 1)
        teacher_probs: [seq_len, 256] float32 — distribución marginalizada byte 0
        teacher_mask:  [seq_len] bool — True donde hay señal teacher

    Document packing: documentos concatenados con patch separador [0,0,0,0].
    El packing se hace en el script de byte_marginalization, no acá.
    """

    def __init__(
        self,
        data_path: str | None = None,
        seq_len: int = 2048,
        dry_run_samples: int = 0,
        teacher_coverage: float = 0.25,
    ):
        """
        Args:
            data_path:        ruta a HF dataset, directorio, .parquet, o repo HF Hub.
                              Mutuamente excluyente con dry_run_samples.
            seq_len:          patches por secuencia (2048 patches = 8192 bytes de contexto)
            dry_run_samples:  si > 0, genera datos aleatorios en vez de cargar datos reales.
                              Util para testear el training loop sin necesitar el dataset.
            teacher_coverage: fraccion de posiciones con senal teacher en dry_run (~25%)
        """
        self.seq_len = seq_len

        if dry_run_samples > 0:
            self.mode = 'dry_run'  # Datos sinteticos para testing
            self.n_samples = dry_run_samples
            self.teacher_coverage = teacher_coverage
        elif data_path is not None:
            self.mode = 'hf'  # Datos reales desde HuggingFace/Parquet
            self._load_hf(data_path)
        else:
            raise ValueError("Necesitas data_path o dry_run_samples > 0")

    def _load_hf(self, data_path: str):
        """Carga desde directorio HF o Parquet."""
        from datasets import load_from_disk, load_dataset
        p = Path(data_path)
        if p.is_dir() and (p / 'dataset_info.json').exists():
            self.dataset = load_from_disk(data_path)
        elif p.suffix == '.parquet':
            self.dataset = load_dataset('parquet', data_files=str(p), split='train')
        else:
            # Intentar como repo de HuggingFace Hub
            self.dataset = load_dataset(data_path, split='train')

    def __len__(self) -> int:
        if self.mode == 'dry_run':
            return self.n_samples
        return len(self.dataset)

    def __getitem__(self, idx: int) -> dict[str, Tensor]:
        if self.mode == 'dry_run':
            return self._get_dry_run_item()
        return self._get_hf_item(idx)

    def _get_dry_run_item(self) -> dict[str, Tensor]:
        """Sample aleatorio para testing del loop."""
        S = self.seq_len
        patches = torch.randint(0, 256, (S, PATCH_SIZE), dtype=torch.long)
        targets = torch.randint(0, 256, (S, PATCH_SIZE), dtype=torch.long)

        # Teacher: distribución sparse
        teacher_probs = torch.zeros(S, VOCAB_SIZE)
        teacher_mask = torch.rand(S) < self.teacher_coverage

        # Llenar posiciones con señal teacher
        n_with_teacher = teacher_mask.sum().item()
        if n_with_teacher > 0:
            # ~8 bytes activos por posición, softmax random
            for i in teacher_mask.nonzero(as_tuple=True)[0]:
                n_active = torch.randint(3, 12, (1,)).item()
                active = torch.randperm(256)[:n_active]
                teacher_probs[i, active] = torch.softmax(torch.randn(n_active), dim=0)

        return {
            'patches': patches,
            'targets': targets,
            'teacher_probs': teacher_probs,
            'teacher_mask': teacher_mask,
        }

    def _get_hf_item(self, idx: int) -> dict[str, Tensor]:
        """Carga sample desde HuggingFace dataset.

        Soporta tanto arrays nativos (HF Dataset con features tipadas) como
        JSON strings (formato de byte_marginalization.py en Parquet crudo).
        """
        item = self.dataset[idx]

        raw_patches = item['patches']
        if isinstance(raw_patches, str):
            raw_patches = json.loads(raw_patches)
        patches = torch.as_tensor(raw_patches, dtype=torch.long)

        raw_targets = item['targets']
        if isinstance(raw_targets, str):
            raw_targets = json.loads(raw_targets)
        targets = torch.as_tensor(raw_targets, dtype=torch.long)

        if 'teacher_probs' in item and item['teacher_probs'] is not None:
            raw_tp = item['teacher_probs']
            if isinstance(raw_tp, str):
                raw_tp = json.loads(raw_tp)
            teacher_probs = torch.as_tensor(raw_tp, dtype=torch.float32)
        else:
            teacher_probs = torch.zeros(patches.shape[0], VOCAB_SIZE)

        if 'teacher_mask' in item and item['teacher_mask'] is not None:
            raw_tm = item['teacher_mask']
            if isinstance(raw_tm, str):
                raw_tm = json.loads(raw_tm)
            teacher_mask = torch.as_tensor(raw_tm, dtype=torch.bool)
        else:
            teacher_mask = torch.zeros(patches.shape[0], dtype=torch.bool)

        return {
            'patches': patches,
            'targets': targets,
            'teacher_probs': teacher_probs,
            'teacher_mask': teacher_mask,
        }


# =============================================================================
# 3. LR Schedule
# =============================================================================

def get_lr(
    step: int,
    warmup_steps: int = 2000,
    total_steps: int = 100_000,
    max_lr: float = 3e-4,
    min_lr: float = 3e-5,
) -> float:
    """
    Cosine annealing con warmup lineal.

    Args:
        step:         step actual del training (0-indexed)
        warmup_steps: pasos de warmup lineal (0 -> max_lr)
        total_steps:  pasos totales de training
        max_lr:       learning rate maximo alcanzado al final del warmup
        min_lr:       learning rate minimo al final del coseno (10% del max)
    Returns:
        learning rate para este step

    Curva:
        [0, warmup)       -> rampa lineal de 0 a max_lr
        [warmup, total)   -> coseno de max_lr a min_lr
        [total, ...)      -> min_lr constante
    """
    if step < warmup_steps:
        return max_lr * step / max(warmup_steps, 1)  # Rampa lineal
    if step >= total_steps:
        return min_lr  # Piso
    # Cosine decay: empieza en max_lr, termina en min_lr
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return min_lr + 0.5 * (max_lr - min_lr) * (1.0 + math.cos(math.pi * progress))


# =============================================================================
# 4. Param Groups
# =============================================================================

def build_param_groups(model: nn.Module) -> list[dict]:
    """
    Separa parámetros en 4 grupos con diferente weight decay y LR.

    - BitLinear weights + embedding projection: wd=0.1
    - FPResidual, RMSNorm, biases, byte_embedding, BitEmbedding scale: wd=0.0
    - Head weights (nn.Linear o BitHead): wd=0.01
    - PFNet weights: wd=0.05, lr=base_lr*1.5 (higher LR for faster convergence)

    Compatible con CPU1ModelV2.
    """
    decay_params = []       # BitLinear weights, embedding proj
    no_decay_params = []    # FPResidual, norms, biases, byte embedding
    head_params = []        # prediction heads
    pfnet_params = []       # PFNet weights (separate LR)

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        # PFNet parameters get their own group
        if '.pfnet.' in name:
            pfnet_params.append(param)
        # v1 usa head_0..head_3, v2 usa byte_heads.0..byte_heads.3
        elif ('head_' in name or 'byte_heads' in name) and 'weight' in name:
            head_params.append(param)
        elif any(k in name for k in ('fp_A', 'fp_B', 'norm', 'bias',
                                      'byte_embedding', 'byte_embed',
                                      'output_scale', '.scale')):
            # FPResidual, RMSNorm, biases, lookup/BitEmbedding, scales — sin decay
            no_decay_params.append(param)
        else:
            # BitLinear weights (q/k/v/o_proj, gate/up/down_proj, forget/cand/output_proj)
            # + embedding.proj / patch_proj
            decay_params.append(param)

    return [
        {'params': decay_params, 'weight_decay': 0.1},
        {'params': no_decay_params, 'weight_decay': 0.0},
        {'params': head_params, 'weight_decay': 0.01},
        {'params': pfnet_params, 'weight_decay': 0.05, 'lr_scale': 1.5},
    ]


# =============================================================================
# 5. Checkpoint
# =============================================================================

def save_checkpoint(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    step: int,
    loss: float,
    path: str,
    upload_to_hf: bool = False,
    hf_repo: str = "Cukinator/cpu1-checkpoints",
):
    """
    Guarda checkpoint a disco y opcionalmente sube a HuggingFace Hub.

    Args:
        model:        modelo (puede ser DDP wrapper o modelo directo)
        optimizer:    estado del optimizador (momentum, etc.)
        scaler:       estado del GradScaler (escala FP16 actual)
        step:         step de training actual
        loss:         ultimo loss registrado
        path:         ruta local donde guardar el .pt
        upload_to_hf: si True, sube el .pt al repo de HuggingFace
        hf_repo:      repo HF destino (default: Cukinator/cpu1-checkpoints)

    Nota: guarda model/optimizer/scaler state_dicts para poder resumir training.
    Si el modelo esta wrapeado en DDP, extrae model.module para guardar sin el wrapper.
    """
    # .module unwraps DDP para obtener el modelo real
    state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
    checkpoint = {
        'step': step,
        'model_state_dict': state,
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'loss': loss,
    }
    torch.save(checkpoint, path)

    if upload_to_hf:
        try:
            from huggingface_hub import HfApi
            api = HfApi()
            api.upload_file(
                path_or_fileobj=path,
                path_in_repo=Path(path).name,
                repo_id=hf_repo,
                repo_type="model",
            )
        except Exception as e:
            print(f"[WARN] No se pudo subir checkpoint a HF Hub: {e}")


def load_checkpoint(
    path: str,
    model: nn.Module,
    optimizer: torch.optim.Optimizer | None = None,
    scaler: torch.amp.GradScaler | None = None,
) -> int:
    """
    Carga checkpoint y retorna el step desde donde resumir.

    Args:
        path:      ruta al archivo .pt
        model:     modelo destino (puede ser DDP wrapper)
        optimizer: si se pasa, restaura el estado del optimizador (momentum, etc.)
        scaler:    si se pasa, restaura el estado del GradScaler
    Returns:
        step (int) desde el cual resumir el training
    """
    checkpoint = torch.load(path, map_location='cpu', weights_only=False)

    target = model.module if hasattr(model, 'module') else model
    target.load_state_dict(checkpoint['model_state_dict'])

    if optimizer and 'optimizer_state_dict' in checkpoint:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    if scaler and 'scaler_state_dict' in checkpoint:
        scaler.load_state_dict(checkpoint['scaler_state_dict'])

    return checkpoint.get('step', 0)


# =============================================================================
# 6. DDP Setup
# =============================================================================

def setup_ddp() -> tuple[int, int]:
    """
    Inicializa DDP para 2xT4 en Kaggle (PCIe only, sin NVLink).

    Returns:
        (local_rank, world_size): indice de GPU local y cantidad total de GPUs

    Las variables de entorno NCCL desactivan P2P e InfiniBand porque las T4
    de Kaggle estan conectadas por PCIe, no por NVLink. Sin esto, NCCL intenta
    usar P2P que falla silenciosamente y mata el rendimiento.
    """
    os.environ.setdefault("NCCL_P2P_DISABLE", "1")    # Sin peer-to-peer GPU directo
    os.environ.setdefault("NCCL_IB_DISABLE", "1")     # Sin InfiniBand
    os.environ.setdefault("NCCL_SOCKET_IFNAME", "lo") # Usar loopback para comunicacion

    dist.init_process_group(backend="nccl")
    local_rank = int(os.environ["LOCAL_RANK"])  # Seteado por torchrun
    torch.cuda.set_device(local_rank)

    return local_rank, dist.get_world_size()


# =============================================================================
# 7. Training Loop
# =============================================================================

def train(args: argparse.Namespace):
    """Training loop principal."""
    # --- Setup dispositivo ---
    is_ddp = 'LOCAL_RANK' in os.environ
    if is_ddp:
        local_rank, world_size = setup_ddp()
        device = torch.device(f'cuda:{local_rank}')
        is_main = (local_rank == 0)
    else:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        local_rank = 0
        world_size = 1
        is_main = True

    if is_main:
        print(f"Device: {device} | World size: {world_size}")

    # --- Modelo ---
    model = CPU1ModelV2({
        'use_gradient_checkpointing': args.gradient_checkpointing,
    }).to(device)

    if is_ddp:
        model = DDP(model, device_ids=[local_rank])

    if is_main:
        n_params = sum(p.numel() for p in model.parameters())
        print(f"Parámetros: {n_params:,}")

    # --- Optimizer ---
    param_groups = build_param_groups(model)
    optimizer = torch.optim.AdamW(
        param_groups,
        lr=args.max_lr,
        betas=(0.9, 0.95),
        eps=1e-8,
    )

    # --- Mixed precision (FP16 en T4, deshabilitado en CPU) ---
    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    # --- Dataset ---
    if args.dry_run:
        dataset = DistillationDataset(
            dry_run_samples=args.dry_run_samples,
            seq_len=args.seq_len,
        )
    else:
        dataset = DistillationDataset(
            data_path=args.data_path,
            seq_len=args.seq_len,
        )

    sampler = DistributedSampler(dataset, shuffle=True) if is_ddp else None
    dataloader = DataLoader(
        dataset,
        batch_size=args.batch_per_gpu,
        sampler=sampler,
        shuffle=(sampler is None),
        num_workers=args.num_workers,
        pin_memory=use_amp,
        drop_last=True,
    )

    # --- Resume ---
    start_step = 0
    if args.resume_from:
        start_step = load_checkpoint(args.resume_from, model, optimizer, scaler)
        if is_main:
            print(f"Resumiendo desde step {start_step}")

    # --- Config ---
    grad_accum = args.gradient_accumulation_steps
    total_steps = args.total_steps
    effective_batch = args.batch_per_gpu * grad_accum * world_size

    if is_main:
        print(f"Batch efectivo: {effective_batch} "
              f"({args.batch_per_gpu} × {grad_accum} accum × {world_size} GPUs)")
        print(f"Steps: {start_step} ->{total_steps}")
        print(f"Seq len: {args.seq_len} patches = {args.seq_len * PATCH_SIZE} bytes")

    # --- Training loop ---
    model.train()
    data_iter = iter(dataloader)
    epoch = 0
    log_timer = time.time()
    metrics: dict[str, float] = {}
    last_loss = 0.0

    for step in range(start_step, total_steps):
        # LR schedule
        lr = get_lr(step, args.warmup_steps, total_steps, args.max_lr, args.min_lr)
        for pg in optimizer.param_groups:
            pg['lr'] = lr * pg.get('lr_scale', 1.0)

        optimizer.zero_grad()
        should_log = (step % args.log_every == 0) and is_main

        for micro_step in range(grad_accum):
            # Obtener batch (reciclar dataloader al agotarse)
            try:
                batch = next(data_iter)
            except StopIteration:
                epoch += 1
                if is_ddp:
                    sampler.set_epoch(epoch)
                data_iter = iter(dataloader)
                batch = next(data_iter)

            patches = batch['patches'].to(device, non_blocking=True)
            targets = batch['targets'].to(device, non_blocking=True)
            teacher_probs = batch['teacher_probs'].to(device, non_blocking=True)
            teacher_mask = batch['teacher_mask'].to(device, non_blocking=True)

            # no_sync() evita sincronizar gradientes entre GPUs en micro-steps intermedios.
            # Solo en el ultimo micro-step se hace all-reduce (suma gradientes de ambas GPUs).
            # Esto ahorra ancho de banda PCIe, critico sin NVLink.
            is_last_micro = (micro_step == grad_accum - 1)
            sync_ctx = contextlib.nullcontext() if (is_last_micro or not is_ddp) else model.no_sync()

            with sync_ctx:
                amp_ctx = torch.amp.autocast('cuda', dtype=torch.float16) if use_amp else contextlib.nullcontext()
                with amp_ctx:
                    logits, _ = model(patches)
                    loss, metrics = distillation_loss(
                        logits, targets, teacher_probs, teacher_mask,
                        temperature=args.temperature,
                        compute_metrics=(should_log and is_last_micro),
                    )
                    if is_last_micro:
                        last_loss = loss.item()
                    loss = loss / grad_accum

                scaler.scale(loss).backward()

        # Gradient clipping
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()

        # Logging
        if should_log:
            elapsed = time.time() - log_timer
            steps_since_log = max(args.log_every, 1)
            patches_per_sec = (args.batch_per_gpu * args.seq_len * grad_accum * steps_since_log) / max(elapsed, 1e-6)

            parts = [
                f"Step {step:6d}/{total_steps}",
                f"Loss: {metrics.get('loss_total', 0):.4f}",
                f"H0-KL: {metrics.get('loss_head0_kl', 0):.4f}",
                f"H0-CE: {metrics.get('loss_head0_ce', 0):.4f}",
                f"H123: {metrics.get('loss_heads_123_ce', 0):.4f}",
                f"Tcov: {metrics.get('teacher_coverage', 0):.1%}",
                f"‖g‖: {grad_norm:.2f}",
                f"LR: {lr:.2e}",
                f"P/s: {patches_per_sec:.0f}",
            ]
            if use_amp:
                gpu_gb = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
                parts.append(f"VRAM: {gpu_gb:.1f}G")

            print(" | ".join(parts))
            log_timer = time.time()

        # Checkpoint
        if step > 0 and step % args.save_every == 0 and is_main:
            ckpt_path = str(Path(args.output_dir) / f"checkpoint_{step}.pt")
            save_checkpoint(
                model, optimizer, scaler, step,
                last_loss,
                ckpt_path,
                upload_to_hf=args.upload_hf,
            )
            print(f"  Checkpoint: {ckpt_path}")

    # --- Final ---
    if is_main:
        ckpt_path = str(Path(args.output_dir) / f"checkpoint_final.pt")
        save_checkpoint(
            model, optimizer, scaler, total_steps,
            last_loss,
            ckpt_path,
            upload_to_hf=args.upload_hf,
        )
        print(f"Training completo. Final: {ckpt_path}")

    if is_ddp:
        dist.destroy_process_group()


# =============================================================================
# 8. CLI
# =============================================================================

def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description="Training CPU-1 con distillation loss")

    # Data
    p.add_argument('--data-path', type=str, default=None,
                   help="Ruta a HF dataset, directorio, o archivo .parquet")
    p.add_argument('--dry-run', action='store_true',
                   help="Usar datos aleatorios (testing)")
    p.add_argument('--dry-run-samples', type=int, default=1000)
    p.add_argument('--seq-len', type=int, default=2048,
                   help="Patches por secuencia")

    # Training
    p.add_argument('--batch-per-gpu', type=int, default=4)
    p.add_argument('--gradient-accumulation-steps', type=int, default=2)
    p.add_argument('--total-steps', type=int, default=100_000)
    p.add_argument('--warmup-steps', type=int, default=2000)
    p.add_argument('--max-lr', type=float, default=3e-4)
    p.add_argument('--min-lr', type=float, default=3e-5)
    p.add_argument('--temperature', type=float, default=4.0)
    p.add_argument('--gradient-checkpointing', action='store_true')

    # Output
    p.add_argument('--output-dir', type=str, default='./checkpoints')
    p.add_argument('--save-every', type=int, default=500)
    p.add_argument('--log-every', type=int, default=100)
    p.add_argument('--upload-hf', action='store_true',
                   help="Subir checkpoints a Cukinator/cpu1-checkpoints")

    # Resume
    p.add_argument('--resume-from', type=str, default=None,
                   help="Ruta a checkpoint .pt para resumir")

    # Workers
    p.add_argument('--num-workers', type=int, default=2)

    return p.parse_args()


# =============================================================================
# 9. Tests
# =============================================================================

def test_distillation_loss():
    """Verifica que la loss computa correctamente y es diferenciable."""
    B, S = 2, 8
    logits = tuple(torch.randn(B, S, 256, requires_grad=True) for _ in range(4))
    targets = torch.randint(0, 256, (B, S, 4))

    # Teacher: distribución sparse sobre ~10 bytes
    teacher_probs = torch.zeros(B, S, 256)
    for b in range(B):
        for s in range(S):
            active = torch.randint(0, 256, (8,))
            teacher_probs[b, s, active] = torch.softmax(torch.randn(8), dim=0)
    teacher_mask = torch.rand(B, S) > 0.5

    loss, metrics = distillation_loss(
        logits, targets, teacher_probs, teacher_mask,
        compute_metrics=True,
    )

    assert loss.shape == (), f"Loss debería ser escalar, got {loss.shape}"
    assert loss.requires_grad, "Loss debería requerir gradientes"
    assert not math.isnan(loss.item()), "Loss es NaN"
    assert 'loss_total' in metrics
    assert 'loss_head0_kl' in metrics
    assert 'teacher_coverage' in metrics

    # Verificar backward
    loss.backward()
    for i, l in enumerate(logits):
        assert l.grad is not None, f"Head {i} no recibió gradiente"

    print(f"[OK] distillation_loss: {loss.item():.4f}")
    for k, v in metrics.items():
        print(f"     {k}: {v:.4f}")


def test_param_groups():
    """Verifica que todos los parámetros se asignan a exactamente un grupo (v1 y v2)."""
    for label, model in [
        ("v2", CPU1ModelV2({'d_model': 2048, 'n_layers': 2, 'd_ff': 5504, 'fp_rank': 8})),
    ]:
        groups = build_param_groups(model)

        grouped = sum(p.numel() for g in groups for p in g['params'])
        total = sum(p.numel() for p in model.parameters() if p.requires_grad)

        assert grouped == total, (
            f"[{label}] Parámetros en grupos ({grouped:,}) != total ({total:,}). "
            f"Hay parámetros sin asignar."
        )

        # Verificar que no hay duplicados
        ids = []
        for g in groups:
            for p in g['params']:
                assert id(p) not in ids, f"[{label}] Parámetro duplicado en param groups"
                ids.append(id(p))

        print(f"[OK] Param groups {label} (2 capas):")
        for i, g in enumerate(groups):
            n_params = sum(p.numel() for p in g['params'])
            print(f"     Grupo {i} (wd={g['weight_decay']}): {n_params:,} params, "
                  f"{len(g['params'])} tensores")


def test_dataset_dry_run():
    """Verifica que el dataset dry-run produce shapes correctas."""
    ds = DistillationDataset(dry_run_samples=10, seq_len=16)
    assert len(ds) == 10

    sample = ds[0]
    assert sample['patches'].shape == (16, 4), f"patches: {sample['patches'].shape}"
    assert sample['targets'].shape == (16, 4), f"targets: {sample['targets'].shape}"
    assert sample['teacher_probs'].shape == (16, 256), f"teacher_probs: {sample['teacher_probs'].shape}"
    assert sample['teacher_mask'].shape == (16,), f"teacher_mask: {sample['teacher_mask'].shape}"

    # Verificar que teacher_probs suma ~1 donde hay señal
    mask = sample['teacher_mask']
    if mask.any():
        probs_sum = sample['teacher_probs'][mask].sum(dim=-1)
        assert (probs_sum > 0.99).all(), f"Teacher probs no suman 1: {probs_sum}"

    print(f"[OK] Dataset dry-run: {len(ds)} samples, shapes correctas")


def test_lr_schedule():
    """Verifica el LR schedule."""
    # Warmup
    assert get_lr(0) == 0.0
    assert abs(get_lr(1000, warmup_steps=2000) - 1.5e-4) < 1e-8
    assert abs(get_lr(2000, warmup_steps=2000) - 3e-4) < 1e-8

    # Cosine decay
    lr_mid = get_lr(51000, warmup_steps=2000, total_steps=100_000)
    assert 3e-5 < lr_mid < 3e-4, f"LR mid: {lr_mid}"

    # Final
    lr_end = get_lr(100_000, warmup_steps=2000, total_steps=100_000)
    assert abs(lr_end - 3e-5) < 1e-8, f"LR final: {lr_end}"

    print(f"[OK] LR schedule: warmup ->{get_lr(2000):.2e}, mid ->{lr_mid:.2e}, end ->{lr_end:.2e}")


if __name__ == "__main__":
    # Si no hay argumentos de CLI, correr tests
    import sys
    if len(sys.argv) == 1:
        print("Sin argumentos -> corriendo tests\n")
        print("=" * 60)
        test_distillation_loss()
        print()
        test_param_groups()
        print()
        test_dataset_dry_run()
        print()
        test_lr_schedule()
        print("\n" + "=" * 60)
        print("Todos los tests pasaron.")
        print("\nPara entrenar: python train.py --dry-run --total-steps 10 --log-every 1")
    else:
        args = parse_args()
        os.makedirs(args.output_dir, exist_ok=True)
        train(args)


In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    print("OK: HF_TOKEN configurado")
except Exception as e:
    print("AVISO: HF_TOKEN no encontrado:", e)
    print("  Los checkpoints solo se guardaran localmente en /kaggle/working/")


In [ ]:
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
pathlib.Path(CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
print("Checkpoints ->", CHECKPOINT_DIR)

for f in ["model_v2.py", "train.py"]:
    exists = pathlib.Path("/kaggle/working/" + f).exists()
    print(("OK" if exists else "ERROR") + ": " + f)


## Dry-run -- verificar setup antes de datos reales

In [ ]:
if not GPU_OK:
    print("SKIP: GPU no disponible o incompatible.")
    print("  -> Settings -> Accelerator -> GPU T4 x2 -> Run All")
else:
    cmd_dry = [
        "torchrun",
        "--nproc_per_node", str(n_gpus),
        "--master_port", "12355",
        "/kaggle/working/train.py",
        "--dry-run",
        "--dry-run-samples", "200",
        "--total-steps", "30",
        "--log-every", "5",
        "--batch-per-gpu", "2",
        "--gradient-accumulation-steps", "1",
        "--num-workers", "0",
        "--output-dir", CHECKPOINT_DIR,
    ]
    print("Comando:", " ".join(cmd_dry))
    print("-" * 60)
    result = subprocess.run(cmd_dry)
    print("-" * 60)
    if result.returncode == 0:
        print("OK: Dry-run completado")
    else:
        print("ERROR: Dry-run fallo (exit code", result.returncode, ")")


## Entrenamiento Real

1. Adjuntar dataset (Dataset -> Add Data)
2. Ajustar `DATA_PATH`
3. Descomentar la celda siguiente

In [ ]:
# ENTRENAMIENTO REAL -- ajustar DATA_PATH y descomentar

# DATA_PATH = "/kaggle/input/cpu1-dataset"
# RESUME_FROM = None

# cmd_train = [
#     "torchrun",
#     "--nproc_per_node", str(n_gpus),
#     "--master_port", "12355",
#     "/kaggle/working/train.py",
#     "--data-path", DATA_PATH,
#     "--total-steps", "100000",
#     "--warmup-steps", "2000",
#     "--log-every", "100",
#     "--save-every", "500",
#     "--batch-per-gpu", "4",
#     "--gradient-accumulation-steps", "2",
#     "--gradient-checkpointing",
#     "--num-workers", "4",
#     "--upload-hf",
#     "--output-dir", CHECKPOINT_DIR,
# ]
# # if RESUME_FROM:
# #     cmd_train += ["--resume-from", CHECKPOINT_DIR + "/" + RESUME_FROM]
# result = subprocess.run(cmd_train)
# print("Exit code:", result.returncode)

print("Celda comentada -- descomentar cuando el dataset este listo")
